In [188]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mudha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mudha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mudha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mudha\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [189]:
df = pd.read_csv("Amazon_Reviews.csv", nrows=5000)

print(df.head())
print(df.shape)
print(df.columns)

      Reviewer Name                     Profile Link Country Review Count  \
0        Eugene ath  /users/66e8185ff1598352d6b3701a      US     1 review   
1  Daniel ohalloran  /users/5d75e460200c1f6a6373648c      GB    9 reviews   
2          p fisher  /users/546cfcf1000064000197b88f      GB   90 reviews   
3         Greg Dunn  /users/62c35cdbacc0ea0012ccaffa      AU    5 reviews   
4     Sheila Hannah  /users/5ddbe429478d88251550610e      GB    8 reviews   

                Review Date                  Rating  \
0  2024-09-16T13:44:26.000Z  Rated 1 out of 5 stars   
1  2024-09-16T18:26:46.000Z  Rated 1 out of 5 stars   
2  2024-09-16T21:47:39.000Z  Rated 1 out of 5 stars   
3  2024-09-17T07:15:49.000Z  Rated 1 out of 5 stars   
4  2024-09-16T18:37:17.000Z  Rated 1 out of 5 stars   

                                      Review Title  \
0       A Store That Doesn't Want to Sell Anything   
1           Had multiple orders one turned up and…   
2                      I informed these repr

In [190]:
def convert_sentiment(text):
    text = str(text).lower()
    
    if "good" in text or "excellent" in text or "amazing" in text or "love" in text:
        return "positive"
    elif "bad" in text or "worst" in text or "poor" in text or "hate" in text:
        return "negative"
    else:
        return "neutral"

df["Sentiment"] = df["Review Text"].apply(convert_sentiment)

print(df["Sentiment"].value_counts())

Sentiment
neutral     3506
positive     811
negative     683
Name: count, dtype: int64


In [191]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [192]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def remove_stop_words(text):
    words = word_tokenize(text)
    return " ".join([w for w in words if w not in stop_words])

def lemmatize_text(text):
    words = word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(w) for w in words])

In [193]:
df["clean"] = df["Review Text"].apply(clean_text)
df["clean"] = df["clean"].apply(remove_stop_words)
df["clean"] = df["clean"].apply(lemmatize_text)

print(df[["Review Text", "clean"]].head())

                                         Review Text  \
0  I registered on the website, tried to order a ...   
1  Had multiple orders one turned up and driver h...   
2  I informed these reprobates that I WOULD NOT B...   
3  I have bought from Amazon before and no proble...   
4  If I could give a lower rate I would! I cancel...   

                                               clean  
0  registered website tried order laptop entered ...  
1  multiple order one turned driver phone door nu...  
2  informed reprobate would going visit sick rela...  
3  bought amazon problem happy service price amaz...  
4  could give lower rate would cancelled amazon p...  


In [194]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["clean"])

In [195]:
y = df["Sentiment"]

In [196]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["clean"])



y = df["Sentiment"]

In [197]:
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

In [198]:
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

In [199]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)

In [200]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)
y_pred_dt = dt.predict(X_test_tfidf)

In [201]:
def evaluate(y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    return acc, pre, rec, f1

In [202]:
def evaluate(y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    return acc, pre, rec, f1
lr_scores = evaluate(y_test, y_pred_lr)
nb_scores = evaluate(y_test, y_pred_nb)
dt_scores = evaluate(y_test, y_pred_dt)

In [203]:
print("\nDetailed Results")

print("\nLogistic Regression")
print(lr_scores)

print("\nNaive Bayes")
print(nb_scores)

print("\nDecision Tree")
print(dt_scores)


Detailed Results

Logistic Regression
(0.844, np.float64(0.8669382797530338), np.float64(0.844), np.float64(0.8227728923867503))

Naive Bayes
(0.707, np.float64(0.766865055387714), np.float64(0.707), np.float64(0.5917948342829764))

Decision Tree
(0.983, np.float64(0.9830257194244605), np.float64(0.983), np.float64(0.983))
